In [1]:
!pip install pandas requests python-dotenv openai google-genai

In [2]:
!mkdir prediction_market_monitor

In [3]:
!mkdir data reports src

# Config file, control for search term

Search words you want to find. You can change the keyword for searching here

In [4]:
%%writefile src/config.py

KEYWORDS = [
    "prediction markets",
    "prediction market",
    "Polymarket",
    'polymarket',
    "Kalshi",
    "event contracts",
    "forecasting markets",
    "information aggregation markets",
]

DATABASE_PATH = "data/papers.csv"
REPORTS_DIR = "reports"

Writing src/config.py


# Search Papers (Most relevant + Newest one)

Here, we only pick new papers. Thus, the old classical one will not appear on the search list. Delete the line "sort": "publication_date:desc" and     "sort": "published",
    "order": "desc" to collect also old paper


In [5]:
%%writefile src/search_sources.py

import time
import requests
from datetime import datetime

TODAY = datetime.today().strftime("%Y-%m-%d")


def search_openalex(query, rows=20):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": rows,
        "sort": "publication_date:desc" # Pick newest paper
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"OpenAlex error for {query}: {e}")
        return []

    papers = []

    for item in response.json().get("results", []):
        authors = []

        for authorship in item.get("authorships", []):
            author = authorship.get("author", {})
            if author.get("display_name"):
                authors.append(author["display_name"])

        doi = item.get("doi", "")
        if doi:
            doi = doi.replace("https://doi.org/", "")

        primary_location = item.get("primary_location") or {}
        source = primary_location.get("source") or {}

        papers.append({
            "source": "OpenAlex",
            "query": query,
            "title": item.get("display_name", ""),
            "authors": ", ".join(authors),
            "year": item.get("publication_year", ""),
            "publication_date": item.get("publication_date", ""),
            "venue": source.get("display_name", ""),
            "abstract": "",
            "doi": doi,
            "url": item.get("id", ""),
            "date_found": TODAY
        })

    return papers


def search_crossref(query, rows=20):
    url = "https://api.crossref.org/works"

    params = {
        "query.bibliographic": query,
        "rows": rows,
        "sort": "published",
        "order": "desc" # Pick newest paper
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Crossref error for {query}: {e}")
        return []

    papers = []

    for item in response.json().get("message", {}).get("items", []):
        title_list = item.get("title", [""])
        venue_list = item.get("container-title", [""])

        date_parts = (
            item.get("published-print")
            or item.get("published-online")
            or item.get("published")
            or {}
        ).get("date-parts", [[None]])

        year = date_parts[0][0] if date_parts and date_parts[0] else ""

        papers.append({
            "source": "Crossref",
            "query": query,
            "title": title_list[0] if title_list else "",
            "authors": ", ".join(
                f"{a.get('given', '')} {a.get('family', '')}".strip()
                for a in item.get("author", [])
            ),
            "year": year,
            "publication_date": "",
            "venue": venue_list[0] if venue_list else "",
            "abstract": item.get("abstract", ""),
            "doi": item.get("DOI", ""),
            "url": item.get("URL", ""),
            "date_found": TODAY
        })

    return papers


def search_all_sources(query, rows=20):
    papers = []

    print(f"Searching OpenAlex: {query}")
    papers.extend(search_openalex(query, rows=rows))

    time.sleep(1)

    print(f"Searching Crossref: {query}")
    papers.extend(search_crossref(query, rows=rows))

    time.sleep(10)

    return papers

Writing src/search_sources.py


In [6]:
from src.search_sources import search_all_sources; search_all_sources("prediction markets", rows=3)

Searching OpenAlex: prediction markets
Searching Crossref: prediction markets


[{'source': 'OpenAlex',
  'query': 'prediction markets',
  'title': 'Exploring the Entrepreneurial Intentions of Academic Inventors in an Emerging Context: The Case of Morocco',
  'authors': 'Mounia Diamane',
  'year': 2028,
  'publication_date': '2028-01-01',
  'venue': 'Cairn.info',
  'abstract': '',
  'doi': None,
  'url': 'https://openalex.org/W7137390646',
  'date_found': '2026-06-09'},
 {'source': 'OpenAlex',
  'query': 'prediction markets',
  'title': 'Unveiling the Dynamics of Super Apps: Adoption Factors and Strategic Insights across Key Industries',
  'authors': 'Jan Budinský, Mohit Srivastava, Heřman Kopkáně, Ladislav Tyll, Ladislav Tyll',
  'year': 2027,
  'publication_date': '2027-01-01',
  'venue': 'Cairn.info',
  'abstract': '',
  'doi': None,
  'url': 'https://openalex.org/W7162702746',
  'date_found': '2026-06-09'},
 {'source': 'OpenAlex',
  'query': 'prediction markets',
  'title': 'Multidisciplinary Practices (MDPs) in Elder Law',
  'authors': 'Bryan Koslow',
  'year

# Database + quality filter

We filter only good journal/working paper. Add or delete the papers here

In [8]:
%%writefile src/database.py

import os
import pandas as pd


def load_database(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame()


def make_paper_id(row):
    doi = str(row.get("doi", "")).lower().strip()
    title = str(row.get("title", "")).lower().strip()

    if doi and doi != "nan":
        return doi

    return title


def remove_duplicates(df):
    if df.empty:
        return df

    df["paper_id"] = df.apply(make_paper_id, axis=1)
    df = df[df["paper_id"] != ""]
    return df.drop_duplicates(subset=["paper_id"])


def filter_quality_sources(df):
    if df.empty:
        return df

    good_journals = [
    "journal of finance",
    "the journal of finance",
    "journal of financial economics",
    "review of financial studies",
    "the review of financial studies",
    "management science",
    "review of finance",
    "journal of financial and quantitative analysis",
]
    working_paper_series = [
        "nber working paper",
        "bis working paper",
        'bis',
        "nber working papers",
        "cepr discussion paper",
        "cepr discussion papers",
        "iza discussion paper",
        "iza discussion papers",
        "ssrn electronic journal",
        "ssrn",
        "federal reserve",
        "fed working paper",
        "bank of england working paper",
        "ecb working paper",
        "imf working paper",
        "world bank policy research working paper",
    ]

    venue = df["venue"].fillna("").str.lower()
    source = df["source"].fillna("").str.lower()
    url = df["url"].fillna("").str.lower()

    keep_good_journal = venue.apply(
        lambda x: any(journal in x for journal in good_journals)
    )

    keep_working_paper = (
        venue.apply(lambda x: any(wp in x for wp in working_paper_series))
        | source.apply(lambda x: any(wp in x for wp in working_paper_series))
        | url.apply(lambda x: any(wp in x for wp in working_paper_series))
    )

    return df[keep_good_journal | keep_working_paper].copy()


def find_new_papers(existing_df, found_df):
    if existing_df.empty:
        return found_df

    existing_ids = set(existing_df["paper_id"])
    return found_df[~found_df["paper_id"].isin(existing_ids)]


def update_database(existing_df, new_df, path):
    combined = pd.concat([existing_df, new_df], ignore_index=True)
    combined = remove_duplicates(combined)

    os.makedirs(os.path.dirname(path), exist_ok=True)
    combined.to_csv(path, index=False)

    return combined

Overwriting src/database.py


# Clean dataset + Gemini retrieved in case the abstract is missing

In [9]:
%%writefile src/cleaning.py

import os
import time
import pandas as pd
from google import genai


def clean_papers(df):
    # 1. Identify duplicate titles
    is_duplicate = df.duplicated(subset=["title"], keep=False)

    # 2. Mark rows where authors contain numbers
    has_digits = df["authors"].astype(str).str.contains(r"\d+", na=False)

    # 3. Mark rows where abstract is missing
    if "abstract" in df.columns:
        has_null_abstract = df["abstract"].isna()
    else:
        df["abstract"] = pd.NA
        has_null_abstract = df["abstract"].isna()

    # 4. Remove duplicate rows only if they are bad
    df_cleaned = df[~(is_duplicate & (has_digits | has_null_abstract))].copy()

    # 5. If duplicates still remain, keep the first one
    df_cleaned = df_cleaned.drop_duplicates(subset=["title"], keep="first")

    return df_cleaned


def fill_missing_abstracts_with_gemini(df):
    # Get Gemini API key from GitHub secret or local environment
    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        print("No GEMINI_API_KEY found. Skipping Gemini abstract step.")
        return df

    client = genai.Client(api_key=api_key)

    for index, row in df.iterrows():
        if pd.isna(row.get("abstract")) or row.get("abstract") == "":
            print(f"Fetching abstract: {row['title']}")

            prompt = (
                f"Provide only the academic abstract for the paper titled "
                f"'{row['title']}' by {row['authors']}. "
                f"Do not include any other text."
            )

            try:
                response = client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt
                )

                df.at[index, "abstract"] = response.text.strip()

                # You can adjust this if Gemini rate limits you
                time.sleep(10)

            except Exception as e:
                print(f"Skipped {row['title']} due to error: {e}")

    # Remove JATS tags if they appear
    df["abstract"] = df["abstract"].astype(str).str.replace(
        r"</?jats:p>",
        "",
        regex=True
    )

    return df

Writing src/cleaning.py


# Report generator

In [10]:
%%writefile src/report.py

import os
from datetime import date


def generate_report(new_papers, reports_dir):
    os.makedirs(reports_dir, exist_ok=True)

    today = date.today().strftime("%Y-%m-%d")
    path = f"{reports_dir}/weekly_digest_{today}.md"

    with open(path, "w", encoding="utf-8") as f:
        f.write("# Weekly Prediction Market Literature Digest\n\n")
        f.write(f"Date: {today}\n\n")

        if new_papers.empty:
            f.write("No new high-quality papers found this week.\n")
            return path

        for _, row in new_papers.iterrows():
            f.write(f"## {row['title']}\n\n")
            f.write(f"**Source:** {row['source']}\n\n")
            f.write(f"**Authors:** {row['authors']}\n\n")
            f.write(f"**Year:** {row['year']}\n\n")
            f.write(f"**Venue:** {row['venue']}\n\n")
            f.write(f"**DOI:** {row['doi']}\n\n")
            f.write(f"**URL:** {row['url']}\n\n")
            f.write("---\n\n")

    return path

Writing src/report.py


# Main weekly script

In [11]:
%%writefile run_weekly.py

import pandas as pd

from src.config import KEYWORDS, DATABASE_PATH, REPORTS_DIR
from src.search_sources import search_all_sources
from src.database import (
    load_database,
    remove_duplicates,
    filter_quality_sources,
    find_new_papers,
    update_database
)
from src.report import generate_report
from src.cleaning import clean_papers, fill_missing_abstracts_with_gemini


def main():
    all_results = []

    for keyword in KEYWORDS:
        print(f"\nSearching keyword: {keyword}")
        results = search_all_sources(keyword, rows=10)
        print(f"Found {len(results)} papers")
        all_results.extend(results)

    found_df = pd.DataFrame(all_results)

    found_df = remove_duplicates(found_df)
    print(f"Total unique papers before quality filter: {len(found_df)}")

    found_df = filter_quality_sources(found_df)
    print(f"Total papers after quality filter: {len(found_df)}")

    existing_df = load_database(DATABASE_PATH)

    if not existing_df.empty:
        existing_df = remove_duplicates(existing_df)

    new_df = find_new_papers(existing_df, found_df)
    print(f"New papers this run: {len(new_df)}")

    # Combine old papers + new papers first
    combined_df = pd.concat([existing_df, new_df], ignore_index=True)

    # Clean duplicate/bad rows
    combined_df = clean_papers(combined_df)

    # Fill missing abstracts with Gemini
    combined_df = fill_missing_abstracts_with_gemini(combined_df)

    # Save final cleaned database
    combined_df.to_csv(DATABASE_PATH, index=False)

    # Generate report only for the new papers
    report_path = generate_report(new_df, REPORTS_DIR)

    print(f"Report created: {report_path}")


if __name__ == "__main__":
    main()

Writing run_weekly.py


In [12]:
%%writefile requirements.txt
pandas
requests
python-dotenv
google-genai

Writing requirements.txt


In [13]:
!pip install -r requirements.txt

In [14]:
!python run_weekly.py


Searching keyword: prediction markets
Searching OpenAlex: prediction markets
Searching Crossref: prediction markets
Found 20 papers

Searching keyword: prediction market
Searching OpenAlex: prediction market
Searching Crossref: prediction market
Found 20 papers

Searching keyword: Polymarket
Searching OpenAlex: Polymarket
Searching Crossref: Polymarket
Found 20 papers

Searching keyword: polymarket
Searching OpenAlex: polymarket
Searching Crossref: polymarket
Found 20 papers

Searching keyword: Kalshi
Searching OpenAlex: Kalshi
Searching Crossref: Kalshi
Found 19 papers

Searching keyword: event contracts
Searching OpenAlex: event contracts
Searching Crossref: event contracts
Found 20 papers

Searching keyword: forecasting markets
Searching OpenAlex: forecasting markets
Searching Crossref: forecasting markets
Found 20 papers

Searching keyword: information aggregation markets
Searching OpenAlex: information aggregation markets
Searching Crossref: information aggregation markets
Found 

In [15]:
import pandas as pd

df = pd.read_csv("data/papers.csv")
df[["title", "authors", "year", "venue", "source", "url"]].head(20)

,title,authors,year,venue,source,url
0,Polymarket-v1 Database,"Boka Qin, Rui Yang",2026,arXiv (Cornell University),OpenAlex,https://openalex.org/W7163720473
1,Adaptive Auto-Harness: Sustained Self-Improvem...,"Zewen Liu, Zhan Shi, Yisi Sang, Bing He, Minhu...",2026,ArXiv.org,OpenAlex,https://openalex.org/W7163719751
2,Benchmarking Security Risk Detection and Verif...,"Ismail Hossain, Sai Puppala, Zhuoran Lu, Sajed...",2026,ArXiv.org,OpenAlex,https://openalex.org/W7163597012
3,Resolution-Aware Perpetual Futures on Binary P...,Maksym Nechepurenko,2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6748278
4,Who Wins and Who Loses In Prediction Markets? ...,"Pat Akey, Vincent Gregoire, Nicolas Harvie, Ch...",2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6443103
5,What Is the Alpha on Polymarket? A Methodology...,Mengxiao Wang,2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6625018
6,The Anatomy of a Blockchain Prediction Market:...,"Zichao Yang, Kwok Ping Tsang",2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6336679
7,"The Polymarket Paradox: Manipulation, Whale Co...",Muhammad Noraiz Abid,2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6670638
8,Exploiting Mean-Reversion in Decentralized Pre...,"Radovan Vojtko, Cyril Dujava",2026,NaN,Crossref,https://doi.org/10.2139/ssrn.6726362
9,Kalshi and the Rise of Macro Markets,"Anthony M. Diercks, Jared Dean Katz, Jonathan ...",2026,SSRN Electronic Journal,Crossref,https://doi.org/10.2139/ssrn.6093294
